# P6: is the map used when chaining gets hard? (Gemma 3 12B; preregistered: `docs/preregistration_p6.md`)

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
3. **Runtime → Change runtime type → A100 GPU.**

Then **Runtime → Run all**. Expected time on an A100: about 45–75 minutes.

**If Colab disconnects:** reconnect and press **Run all** again. Finished steps are saved in `MyDrive/geomap/results/p6/`
and skipped on the rerun.

In [ ]:
# 1. GPU check
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
assert gb > 35, f"Only {gb:.0f} GB of GPU memory: pick an A100 (40 GB) runtime"

In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
assert os.path.exists("scripts/p6_hard.py"), "Old bundle: upload the new geomap_colab.zip"
print("already saved:", sorted(os.listdir(f"{RES}/p6")) if os.path.exists(f"{RES}/p6") else "nothing yet")

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' Gemma 3 import)
!pip -q install "transformers==4.57.3" nnsight accelerate
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")
!python -c "from transformers.models.gemma3.modeling_gemma3 import Gemma3ForConditionalGeneration; print('Gemma 3 import OK')"

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
from geomap.checkpoint import notify
notify("[P6] Colab run started (or resumed)")

In [ ]:
# 5. P6: erase the map on 4-5-object scenes (~45-75 min; resumes if interrupted)
!cd /content/geomap && python scripts/p6_hard.py --model gemma3-12b 2>&1 | grep -v -i warning

In [ ]:
# 6. One zip of the P6 results + the results log, on Drive and downloaded here
shutil.make_archive("/content/p6_results", "zip", "/content/geomap", "results/p6")
with zipfile.ZipFile("/content/p6_results.zip", "a") as z:
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
shutil.copy("/content/p6_results.zip", f"{D}/p6_results.zip")
notify("[P6] all done: results in MyDrive/geomap/p6_results.zip")
from google.colab import files
files.download("/content/p6_results.zip")